#Initialization and Loading Library

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("crm_product_pipeline")

logger.info("Starting Silver Layer Transformation")

#Reading From Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")
logger.info(f"Number of rows (raw): {df.count()}")
logger.info(f"Columns: {df.columns}")
df.display()

#Data Quality Check

In [0]:
from pyspark.sql.functions import count, when

# Null check
null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

logger.info("Null counts:")
null_counts.show()

# Duplicate check
dup_count = df.count() - df.dropDuplicates().count()
logger.info(f"Duplicate rows: {dup_count}")

#Data Transformations

##Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

logger.info("String columns trimmed")

##Product Key parsing

In [0]:

df = df.withColumn("cat_id", regexp_replace(substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", substring(col("prd_key"), 7, length(col("prd_key"))))

logger.info("Columns 'cat_id' created and 'prd_key' updated")


In [0]:
df.display()

##Cost column Cleanup

In [0]:
df = df.withColumn("prd_cost", coalesce(col("prd_cost"), lit(0)))

logger.info("Column 'prd_cost' nulls replaced with 0")
display(df)


##Product Line Normalization

In [0]:
logger.info("Applying product line normalization")

df = (
    df.withColumn(
        "prd_line",
        when(upper(col("prd_line")) == "M", lit("Mountain"))
        .when(upper(col("prd_line")) == "R", lit("Road"))
        .when(upper(col("prd_line")) == "S", lit("Other Sales"))
        .when(upper(col("prd_line")) == "T", lit("Touring"))
        .otherwise(lit("n/a"))
    )
)



df.display()

##Date Casting

In [0]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

logger.info("Casting 'prd_start_dt' to DateType")

##Renaming Column

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

logger.info("Columns renamed successfully!")

##Data Quality Check

In [0]:
df.limit(10).display()

#Writing Data to Silver Table

In [0]:
logger.info("Writing data to Silver table")

(
    df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.crm_products")
)

logger.info("Write completed successfully")


##Data quality check

In [0]:

%sql
SELECT * FROM workspace.silver.crm_products LIMIT 10